In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import os

os.chdir('/content/drive/MyDrive')

os.listdir()

['Colab Notebooks',
 '155410_DE1_0_2008_to_2010_Inpatient_Claims_Sample_20.zip',
 'DE1_0_2008_to_2010_Inpatient_Claims_Sample_20.csv',
 'Copy of MSDS Degree Progress  (1).gdoc',
 'Copy of [MAKE A COPY] DS+X Student Waiver & Photo Release (1).docx.gdoc',
 'd_items.gsheet',
 'Copy of [MAKE A COPY] DS+X Student Waiver & Photo Release.docx.gdoc',
 'astika resume BU.pdf',
 'astika bu resume 2 (1).pdf',
 'bio celt.jpeg',
 'Copy of MSDS Degree Progress .gdoc',
 'Astika_Tyagi_Resume_CELT.pdf',
 'device-recall-0001-of-0001.json.zip']

In [4]:
import zipfile

with zipfile.ZipFile('device-recall-0001-of-0001.json.zip', 'r') as zip_ref:
    zip_ref.extractall('recall_data')

In [5]:
os.listdir('recall_data')

['device-recall-0001-of-0001.json']

In [6]:
import json
import pandas as pd
from pandas import json_normalize

with open('recall_data/device-recall-0001-of-0001.json') as f:
    data = json.load(f)

df = json_normalize(data['results'])

print(df.shape)
df.head()

(57511, 35)


,cfres_id,product_res_number,event_date_initiated,event_date_posted,recall_status,res_event_number,product_code,product_description,code_info,recalling_firm,...,openfda.regulation_number,openfda.device_class,address_2,event_date_terminated,firm_fei_number,other_submission_description,k_numbers,pma_numbers,event_date_created,openfda.pma_number
0,218307,Z-1472-2026,2026-01-21,2026-02-26,"Open, Classified",98332,FNM,"Adapt Pump, Model/Catalog Number: 61600200-Ada...",UDI-DI 0084569904914; All serial numbers with ...,Agiliti Health Ellis,...,880.5550,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,218406,Z-1462-2026,2026-01-14,2026-02-24,"Open, Classified",98303,KPL,"PIE PAK\nModels: P2HC-A, P2HC-S, P2HC",All Lots\nUDI:,Edermy LLC,...,876.5220,2,Ste A,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,105069,Z-0389-2012,2011-09-19,2012-01-11,Terminated,60344,LFQO,Magnus Hybrid operating table columns 1180.01A...,"510 k EXEMPT 1180.01A1- Lot #0002, 0004, 0005...",Maquet Inc.,...,NaN,NaN,NaN,2011-12-08,NaN,NaN,NaN,NaN,NaN,NaN
3,105700,Z-0485-2012,2011-11-28,2012-01-11,Terminated,60528,RER,Raymond Laser alignment guide - Model #1096597...,Model #1096597/002,The Raymond Corporation S Canal,...,,N,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,131624,Z-0604-2015,2014-11-17,2014-12-12,Terminated,69838,KOG,"PKG, OBTURATOR, 5.5MM FOR HASSON CANNULA, P/N ...",37302 37858 39543 45622 48347 48903 49147 4914...,Stryker Endoscopy,...,NaN,NaN,NaN,2015-07-17,2936485,NaN,NaN,NaN,NaN,NaN


In [7]:
# inspect shape and columns
print(df.shape)
print(df.columns.tolist())

(57511, 35)
['cfres_id', 'product_res_number', 'event_date_initiated', 'event_date_posted', 'recall_status', 'res_event_number', 'product_code', 'product_description', 'code_info', 'recalling_firm', 'address_1', 'city', 'state', 'postal_code', 'additional_info_contact', 'reason_for_recall', 'root_cause_description', 'action', 'product_quantity', 'distribution_pattern', 'openfda.k_number', 'openfda.registration_number', 'openfda.fei_number', 'openfda.device_name', 'openfda.medical_specialty_description', 'openfda.regulation_number', 'openfda.device_class', 'address_2', 'event_date_terminated', 'firm_fei_number', 'other_submission_description', 'k_numbers', 'pma_numbers', 'event_date_created', 'openfda.pma_number']


In [8]:
# make a working copy
recalls = df.copy()

In [9]:
# standardize column names
recalls.columns = (
    recalls.columns
    .str.strip()
    .str.lower()
    .str.replace(r"[^a-z0-9]+", "_", regex=True)
    .str.strip("_")
)

print(recalls.columns.tolist())

['cfres_id', 'product_res_number', 'event_date_initiated', 'event_date_posted', 'recall_status', 'res_event_number', 'product_code', 'product_description', 'code_info', 'recalling_firm', 'address_1', 'city', 'state', 'postal_code', 'additional_info_contact', 'reason_for_recall', 'root_cause_description', 'action', 'product_quantity', 'distribution_pattern', 'openfda_k_number', 'openfda_registration_number', 'openfda_fei_number', 'openfda_device_name', 'openfda_medical_specialty_description', 'openfda_regulation_number', 'openfda_device_class', 'address_2', 'event_date_terminated', 'firm_fei_number', 'other_submission_description', 'k_numbers', 'pma_numbers', 'event_date_created', 'openfda_pma_number']


In [11]:
# check which columns contain lists
list_cols = [
    col for col in recalls.columns
    if recalls[col].apply(lambda x: isinstance(x, list)).any()
]

print("Columns with list values:")
print(list_cols)

Columns with list values:
['openfda_k_number', 'openfda_registration_number', 'openfda_fei_number', 'k_numbers', 'pma_numbers', 'openfda_pma_number']


In [12]:
# convert list-valued columns to strings so pandas can hash them
for col in list_cols:
    recalls[col] = recalls[col].apply(
        lambda x: "|".join(map(str, x)) if isinstance(x, list) else x
    )

# now remove exact duplicates
recalls = recalls.drop_duplicates()

print("Shape after dropping duplicates:", recalls.shape)

Shape after dropping duplicates: (57511, 35)


In [14]:
print("Total rows:", len(recalls))
print("Unique recall events:", recalls["cfres_id"].nunique())

Total rows: 57511
Unique recall events: 57503


No exact duplicates were detected. The dataset contains 57,511 rows representing 57,503 unique recall events, meaning a few recalls involve multiple products.

In FDA recall data, one recall event can appear multiple times because:

A recall can involve multiple products

Each product appears as a separate row